# Train one LoRA adapter on Colab

Runtime → Change runtime type → **T4 GPU**. Fill in the parameters, then Runtime → Run all.

One run = one adapter for one subject. The adapter (~30 MB) is pushed to a private Hugging Face repo;
the base model is downloaded here and discarded with the session. Training energy is logged with CodeCarbon
and saved inside the adapter repo as `training_summary.json`.

In [ ]:
#@title Parameters
BASE = "Qwen/Qwen2.5-3B-Instruct"   #@param ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B-Instruct"]
SUBJECT = "math"                   #@param ["math", "code", "knowledge"]
HF_USER = "your-hf-username"       #@param {type:"string"}
N_EXAMPLES = 2000                  #@param {type:"integer"}
EPOCHS = 2                         #@param {type:"number"}
RANK = 16                          #@param {type:"integer"}
FOUR_BIT = False                   #@param {type:"boolean"}   # set True for the 7B (QLoRA); keep False for 1.5B/3B so the adapter matches the fp16 base served on the Mac
COUNTRY = "IND"                    #@param {type:"string"}    # for the training-energy CO2 factor

REPO = f"{HF_USER}/{BASE.split('/')[-1].lower()}-lora-{SUBJECT}"
print("adapter will be pushed to:", REPO)

In [ ]:
#@title Install (pinned so the adapter loads on the Mac with the same peft major version)
!pip -q install "transformers>=4.45,<5" "peft>=0.13" "datasets>=2.20" "accelerate" "codecarbon>=2.8" "huggingface_hub" bitsandbytes
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
#@title Log in to Hugging Face (needs a WRITE token; also accept the base model's licence if it is gated)
from huggingface_hub import login
login()

In [ ]:
#@title Get train_lora.py into this session (pick ONE way)
# Way 1 (simplest): Files panel on the left -> upload icon -> upload train_lora.py and questions.csv.
# Way 2: add a cell above this one starting with  %%writefile train_lora.py  and paste the script under it.
# Way 3: set RAW_URL to the raw GitHub URL of the file.
# Uploads disappear when the session ends, so repeat for each new session.
import os
RAW_URL = ""  #@param {type:"string"}
if not os.path.exists("train_lora.py") and RAW_URL:
    !wget -q -O train_lora.py "$RAW_URL"
assert os.path.exists("train_lora.py"), "train_lora.py not found: upload it (Files panel) or paste it with %%writefile"
print("train_lora.py ready;", "questions.csv ready" if os.path.exists("questions.csv") else "questions.csv not uploaded (exclusion filter will be skipped)")

In [ ]:
#@title Mount Drive for checkpoints (optional; lets a disconnected session resume)
USE_DRIVE = True  #@param {type:"boolean"}
OUT = "adapter_out"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/lora_runs/{REPO.split('/')[-1]}"
    os.makedirs(OUT, exist_ok=True)
print("outputs ->", OUT)

In [ ]:
#@title Train (3B on T4: ~1-2 h for 2,000 examples x 2 epochs; 7B QLoRA: ~3-4 h)
# Built as a Python list and run with subprocess: Colab's $var substitution and backslash
# continuations in a ! line mangled the argument list.
import os, subprocess, sys, shlex
cmd = [sys.executable, "-u", "train_lora.py",
       "--base", BASE, "--subject", SUBJECT, "--n", str(N_EXAMPLES), "--epochs", str(EPOCHS),
       "--rank", str(RANK), "--out", OUT, "--push-to", REPO, "--country", COUNTRY]
if FOUR_BIT:
    cmd.append("--four-bit")
if os.path.exists("questions.csv"):      # evaluation bank: matching training rows are dropped
    cmd += ["--exclude", "questions.csv"]
print(" ".join(shlex.quote(c) for c in cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
#@title Train (3B on T4: ~1-2 h for 2,000 examples x 2 epochs; 7B QLoRA: ~3-4 h)
# Built as a Python list and run with subprocess: Colab's $var substitution and backslash
# continuations in a ! line mangled the argument list.
import os, subprocess, sys, shlex
cmd = [sys.executable, "-u", "train_lora.py",
       "--base", BASE, "--subject", SUBJECT, "--n", str(N_EXAMPLES), "--epochs", str(EPOCHS),
       "--rank", str(RANK), "--out", OUT, "--push-to", REPO, "--country", COUNTRY]
if FOUR_BIT:
    cmd.append("--four-bit")
if os.path.exists("questions.csv"):      # evaluation bank: matching training rows are dropped
    cmd += ["--exclude", "questions.csv"]
print(" ".join(shlex.quote(c) for c in cmd), flush=True)
subprocess.run(cmd, check=True)